In [1]:
# Humor Detection using Ollama Models - zero-shot

## Imports

# ```python
import pandas as pd
import ollama

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

tqdm.pandas()
# ```

## Configuration

# ```python
DATA_PATH = "Data/testing/joke_detection(testing).csv"

LABELS = [
    "joke",
    "not_joke"
]

# MODELS = [
#     "qwen2.5",
#     "llama3.1",
#     "mistral",
#     "gemma3",
#     "command-r7b-arabic"
# ]
MODELS = [
    
    # "llama3.3",
    # "mistral-small3.2",
    # "gemma4",
    # "qwen3.6",
    # "deepseek-r1",
    "command-r7b-arabic"
]
# ```

## Prompt

# ```python
def build_prompt(text):

    return f"""
You are an expert in humor detection.

Task:
Determine whether the following Arabic text is intended to be a joke or not.

Labels:
- joke
- not_joke

Rules:
- Return exactly one label.
- Do not explain your answer.
- Output only the label.

Text:
{text}
"""
# ```

## Prediction Function

# ```python
def predict_label(text, model_name):

    try:

        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": build_prompt(text)
                }
            ]
        )

        prediction = response["message"]["content"].strip()

        if prediction in LABELS:
            return prediction

        prediction = prediction.lower()

        for label in LABELS:
            if prediction == label.lower():
                return label

        return "INVALID"

    except Exception:
        return "INVALID"
# ```

## Evaluation

# ```python
df = pd.read_csv(DATA_PATH)

results = []

for model_name in MODELS:

    preds = []

    for text in tqdm(
        df["text"],
        total=len(df),
        desc=model_name
    ):
        preds.append(
            predict_label(text, model_name)
        )

    acc = accuracy_score(df["label"], preds)

    f1 = f1_score(
        df["label"],
        preds,
        average="macro"
    )

    print("\n", "="*60)
    print(model_name)
    print("Accuracy:", round(acc,4))
    print("Macro F1:", round(f1,4))

    print(
        classification_report(
            df["label"],
            preds,
            digits=4
        )
    )

    tmp = df.copy()
    tmp["prediction"] = preds

    tmp.to_csv(
        f"predictions_{model_name}_detection.csv",
        index=False
    )

    results.append({
        "model": model_name,
        "accuracy": acc,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    "macro_f1",
    ascending=False
)
# ```


command-r7b-arabic:   0%|          | 0/507 [00:00<?, ?it/s]


command-r7b-arabic
Accuracy: 0.8205
Macro F1: 0.642
              precision    recall  f1-score   support

        joke     0.8856    0.9042    0.8948       428
    not_joke     0.4143    0.3671    0.3893        79

    accuracy                         0.8205       507
   macro avg     0.6499    0.6356    0.6420       507
weighted avg     0.8121    0.8205    0.8160       507



,model,accuracy,macro_f1
0,command-r7b-arabic,0.820513,0.64203


In [4]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(d["label"], preds))

[[387  41]
 [ 50  29]]


In [3]:
import pandas as pd
import ollama

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

tqdm.pandas()


DATA_PATH = "Data/testing/joke_detection(testing).csv"
TRAIN_PATH = "Data/training/joke_detection(training).csv"

LABELS = ["joke", "not_joke"]

# MODELS = [
#     "qwen2.5",
#     "llama3.1",
#     "mistral",
#     "gemma3",
#     "command-r7b-arabic"
# ]
MODELS = [
    
    # "llama3.3",
    # "mistral-small3.2",
    # "gemma4",
    # "qwen3.6",
    "deepseek-r1"
]
N_SHOTS = 4

df_test = pd.read_csv(DATA_PATH)
df_train = pd.read_csv(TRAIN_PATH)

def build_few_shot_examples(train_df, label, n_shots=2):

    subset = train_df[train_df["label"] == label]

    if len(subset) == 0:
        raise ValueError(f"No training samples found for label: {label}")

    sampled = subset.sample(
        min(n_shots, len(subset)),
        random_state=42
    )

    return [
        (row["text"], row["label"])
        for _, row in sampled.iterrows()
    ]

def build_prompt(text, few_shot_examples):

    example_text = "\n\n".join([
        f"""Text:
{ex_text}

Label:
{ex_label}"""
        for ex_text, ex_label in few_shot_examples
    ])

    return f"""
You are an expert in humor detection.

Task:
Determine whether the following Arabic text is intended to be a joke or not.

Labels:
- joke
- not_joke

Rules:
- Return exactly one label.
- Do not explain your answer.
- Output only the label.

Examples:
{example_text}

Now classify this:

Text:
{text}

Label:
"""


def predict_label(text, model_name, few_shot_examples):

    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": build_prompt(text, few_shot_examples)
                }
            ]
        )

        prediction = response["message"]["content"].strip()

        if prediction in LABELS:
            return prediction

        prediction = prediction.lower()

        for label in LABELS:
            if prediction == label.lower():
                return label

        return "INVALID"

    except Exception:
        return "INVALID"

results = []

for model_name in MODELS:

    preds = []

    for _, row in tqdm(
        df_test.iterrows(),
        total=len(df_test),
        desc=model_name
    ):

        text = row["text"]
        true_label = row["label"]

        # 🔥 CLASS-CONDITIONED FEW-SHOT (oracle)
        few_shot_examples = build_few_shot_examples(
            df_train,
            label=true_label,
            n_shots=N_SHOTS
        )

        preds.append(
            predict_label(text, model_name, few_shot_examples)
        )

    acc = accuracy_score(df_test["label"], preds)

    f1 = f1_score(
        df_test["label"],
        preds,
        average="macro"
    )

    print("\n", "="*60)
    print(model_name)
    print("Accuracy:", round(acc, 4))
    print("Macro F1:", round(f1, 4))

    print(
        classification_report(
            df_test["label"],
            preds,
            digits=4
        )
    )

    tmp = df_test.copy()
    tmp["prediction"] = preds

    tmp.to_csv(
        f"predictions_class_conditioned_fewshot_{model_name}.csv",
        index=False
    )

    results.append({
        "model": model_name,
        "accuracy": acc,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results)

results_df.sort_values("macro_f1", ascending=False)




deepseek-r1:   0%|          | 0/507 [00:00<?, ?it/s]


deepseek-r1
Accuracy: 0.9329
Macro F1: 0.8574
              precision    recall  f1-score   support

        joke     0.9397    0.9836    0.9612       428
    not_joke     0.8814    0.6582    0.7536        79

    accuracy                         0.9329       507
   macro avg     0.9105    0.8209    0.8574       507
weighted avg     0.9306    0.9329    0.9288       507



,model,accuracy,macro_f1
0,deepseek-r1,0.932939,0.857405
